# Assignment 7.1: Music Genre and Composer Classification Using Deep Learning

| <div align="center"> Field </div>| <div align="center"> Details </div> |
|-------|---------|
| **Team Members** | Pros Loung, Nicole Rowley, Reema Eid |
| **Course** | AAI-511 Neural Networks and Deep Learning |
| **FINAL PROJECT** | Music Genre and Composer Classification Using Deep Learning |
| **GitHub Repository** | https://github.com/nicole-rowley-msaai/MS_AAI_511_Neural_Networks_DL |

## Team Contribution Map

| <div align="center"> Team member </div> | <div align="center"> Workstream </div> | <div align="center"> Sections Owned </div> |
|---|---|---|
| **Pros** | | |
| **Nicole** | |  |
| **Reema** |  | |
| **Team** | Final validation, Final Report, and GitHub submission | Notebook 2 final report sections and presentation |

# Introduction

Music is a form of art that is ubiquitous and has a rich history. Different composers have created music with their unique styles and compositions. However, identifying the composer of a particular piece of music can be a challenging task, especially for novice musicians or listeners. The proposed project aims to use deep learning techniques to identify the composer of a given piece of music accurately.

# Objective

The primary objective of this project is to develop a deep learning model that can predict the composer of a given musical score accurately. The project aims to accomplish this objective by using two deep learning techniques: Long Short-Term Memory (LSTM) and Convolutional Neural Network (CNN).

# Methodology

The proposed project will be implemented using the following steps:

1. Data Collection: Use the provided dataset of musical scores and associated composer labels.
2. Data Pre-processing: Convert scores into a model-ready representation (for example MIDI/event sequences) and apply quality checks and augmentation where appropriate.
3. Feature Extraction: Extract relevant musical features such as note patterns, chord progressions, rhythm, and tempo.
4. Model Building: Develop deep learning classifiers using both LSTM and CNN architectures.
5. Model Training: Train the models on preprocessed and feature-engineered data using consistent train/validation/test splits.
6. Model Evaluation: Evaluate performance using accuracy, precision, recall, and F1-score.
7. Model Optimization: Tune hyperparameters and regularization settings to improve generalization.

# Final Project Setup and Pipeline

1. Project Setup
- Define project scope, assumptions, and success criteria.
- Create reproducible environment and dependency list.
- Set random seeds and experiment tracking conventions.

2. Data Intake and Audit
- Load raw music-score data and labels.
- Check class balance, missing labels, and duplicate samples.
- Establish train/validation/test split strategy with stratification.

3. Representation and Preprocessing
- Convert source files to a consistent symbolic representation.
- Normalize sequence length strategy (padding/truncation/windows).
- Build preprocessing pipeline that can be reused for inference.

4. Feature Engineering
- Build two branches of inputs: sequence-based inputs for LSTM and image/grid-like representations for CNN (if applicable).
- Generate optional handcrafted features (tempo, pitch range, interval stats) for comparison baselines.

5. Baseline Models
- Train simple baseline models to set minimum acceptable performance.
- Record baseline metrics and confusion patterns by composer.

6. Deep Model Development
- LSTM track: embedding/sequence layers, recurrent blocks, dropout, dense classifier.
- CNN track: convolution blocks, pooling, normalization, dense classifier.
- Keep model definitions modular to support ablation studies.

7. Training and Validation
- Use early stopping and model checkpointing.
- Run controlled experiments across learning rate, batch size, depth, and dropout.
- Track every run with config, metrics, and notes.

8. Evaluation and Error Analysis
- Evaluate best models on hold-out test set.
- Report accuracy, precision, recall, F1-score, and confusion matrix.
- Analyze frequent misclassifications and class-specific weaknesses.

9. Model Selection and Optimization
- Compare LSTM vs CNN against objective metrics and training stability.
- Select final model using both metric quality and robustness.
- Apply final hyperparameter tuning pass.

10. Packaging and Reproducibility
- Save preprocessing artifacts, label encoder, and model weights.
- Create an inference function for unseen music input.
- Document end-to-end run steps for reproducibility.

11. Final Deliverables
- Final notebook with narrative and visual results.
- Short report: problem, method, results, limitations, future work.
- Optional demo cell: predict composer for a sample input.

# Pipeline Connection Diagram

```mermaid
flowchart TD
    A[1. Project Setup] --> B[2. Load Data and Exploration]
    B --> C[3. Representation and Preprocessing]
    C --> D[4. Feature Engineering]
    D --> E[5. Baseline Models]
    E --> F[6. Deep Model Development<br/>LSTM and CNN Tracks]
    F --> G[7. Training and Validation]
    G --> H[8. Evaluation and Error Analysis]
    H --> I[9. Model Selection and Optimization]
    I --> J[10. Packaging and Reproducibility]
    J --> K[11. Final Deliverables]

    F -. Compare architectures .-> I
    H -. Feedback loop for improvements .-> C
    I -. Retune if needed .-> G
```

## 1.0 Project Setup
### Owner: Pros


In [3]:
# Install libraries
# Use the following command to install the pretty_midi library if not already installed
#!pip install pretty_midi

In [1]:
# Load Library 
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn

## 2. Load Data and Exploration
### Onwer: Pros

- Load raw music-score data and labels.
- Check class balance, missing labels, and duplicate samples.
- Establish train/validation/test split strategy with stratification.

In [9]:
# Load MIDI files from dataset folders (dev/train/test with composer subfolders)
from pathlib import Path
from collections import Counter
import pretty_midi

data_root = Path(r"C:\Users\Loung\USD\AAI-551 Neural Networks\MS_AAI_511_Neural_Networks_DL")
split_dirs = {
    "dev": data_root / "dev",
    "train": data_root / "train",
    "test": data_root / "test",
}

# Only load these composers
target_composers = ["Bach", "Beethoven", "Chopin", "Mozart"]
target_composers_lower = {name.lower() for name in target_composers}

midi_data = {"dev": [], "train": [], "test": []}
load_errors = []

for split, split_dir in split_dirs.items():
    for midi_file in split_dir.rglob("*.mid"):
        composer = midi_file.parent.name
        if composer.lower() not in target_composers_lower:
            continue

        try:
            pm = pretty_midi.PrettyMIDI(str(midi_file))
            midi_data[split].append(
                {
                    "path": str(midi_file),
                    "composer": composer,
                    "midi": pm,
                }
            )
        except Exception as e:
            load_errors.append((str(midi_file), str(e)))

# Convenience lists if you want to keep your previous variable names
midi_dev = [item["midi"] for item in midi_data["dev"]]
midi_train = [item["midi"] for item in midi_data["train"]]
midi_test = [item["midi"] for item in midi_data["test"]]

# Quick verification output
for split in ["dev", "train", "test"]:
    composer_counts = Counter(item["composer"].lower() for item in midi_data[split])
    print(f"{split.upper()} files loaded: {len(midi_data[split])}")
    for composer_name in target_composers:
        print(f"  - {composer_name}: {composer_counts.get(composer_name.lower(), 0)}")

if load_errors:
    print(f"\nFiles with load errors: {len(load_errors)}")
    for file_path, err in load_errors[:5]:
        print(f"  - {file_path}: {err}")

DEV files loaded: 12
  - Bach: 4
  - Beethoven: 0
  - Chopin: 4
  - Mozart: 4
TRAIN files loaded: 124
  - Bach: 42
  - Beethoven: 0
  - Chopin: 41
  - Mozart: 41
TEST files loaded: 12
  - Bach: 4
  - Beethoven: 0
  - Chopin: 4
  - Mozart: 4


## 3. Representation and Preprocessing
### Owner:
- Convert source files to a consistent symbolic representation.
- Normalize sequence length strategy (padding/truncation/windows).
- Build preprocessing pipeline that can be reused for inference.

## 4. Feature Engineering
### Owner:
- Build two branches of inputs: sequence-based inputs for LSTM and image/grid-like representations for CNN (if applicable).
- Generate optional handcrafted features (tempo, pitch range, interval stats) for comparison baselines.

## 5. Baseline Models
### Owner: 
- Train simple baseline models to set minimum acceptable performance.
- Record baseline metrics and confusion patterns by composer.

## 6. Deep Model Development
### Owner: 
- LSTM track: embedding/sequence layers, recurrent blocks, dropout, dense classifier.
- CNN track: convolution blocks, pooling, normalization, dense classifier.
- Keep model definitions modular to support ablation studies.

## 7. Training and Validation
### Owner:
- Use early stopping and model checkpointing.
- Run controlled experiments across learning rate, batch size, depth, and dropout.
- Track every run with config, metrics, and notes.

## 8. Evaluation and Error Analysis
### Owner: 
- Evaluate best models on hold-out test set.
- Report accuracy, precision, recall, F1-score, and confusion matrix.
- Analyze frequent misclassifications and class-specific weaknesses.

## 9. Model Selection and Optimization
### Owner:
- Compare LSTM vs CNN against objective metrics and training stability.
- Select final model using both metric quality and robustness.
- Apply final hyperparameter tuning pass.

## 10. Packaging and Reproducibility
### Owner:
- Save preprocessing artifacts, label encoder, and model weights.
- Create an inference function for unseen music input.
- Document end-to-end run steps for reproducibility.

## 11. Final Deliverables
### Owner:
- Final notebook with narrative and visual results.
- Short report: problem, method, results, limitations, future work.
- Optional demo cell: predict composer for a sample input.